In [ ]:
# !pip install transformers datasets evaluate seqeval pandas torch scikit-learn

import json
import os
# Must be set before tokenizer import
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import sys
from types import NoneType
from typing import Dict, List, Tuple, Union
import numpy as np
import torch
import evaluate as hf_evaluate
import shutil
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import copy

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    XLMRobertaForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    BertForTokenClassification,
    RobertaTokenizerFast
)
import ast
from sklearn.metrics import f1_score, precision_score, recall_score

# Assumes this notebook is run from the encoders/ directory
REPO_ROOT = Path.cwd().parent
sys.path.append(str(Path.cwd()))

from common import (
    LANGS_TO_MODELS, LANGS, PREDICTIONS_OUT_FILENAME, RESULTS_OUT_FILENAME,
    SEEDS, get_cur_run_dir, PREDICTIONS_OUT_FILENAME_PLUS_EXPLICIT_IDIOMS, write_results_to_dir
)

# id10m-jam variant data (download first: python data_generation/download_id10m_jam.py)
DATA_FILE_FOR_LANG = {
    lang: REPO_ROOT / "data_generation" / "id10m_jam" / f"{lang}.json"
    for lang in LANGS
}

# id10m trainset: download from https://github.com/Babelscape/ID10M
# and place each language's TSV files under: data_generation/id10m_trainset/{lang}/
TRAINSET_DIR = REPO_ROOT / "data_generation" / "id10m_trainset"


# ==========================================
# 0. Global Settings & Constants
# ==========================================

GPU = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU

LABEL_LIST = ["O", "B-IDIOM", "I-IDIOM"]
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}

MODEL_TO_TRAIN_ARGS = {
    "FacebookAI/xlm-roberta-base": {
        'per_device_train_batch_size': 8,
        'per_device_eval_batch_size': 8,
        'lr_scheduler_type': 'linear'
    }
}

seqeval_metric = hf_evaluate.load("seqeval")


# ==========================================
# 1. Data Parsing & Helper Functions
# ==========================================

def parse_tsv_data(file_path: Path) -> Tuple[List[List[str]], List[List[int]]]:
    """
    Parses a BIO-tagged .tsv file with 'word' and 'label' columns.
    Returns (list of token lists, list of label-id lists).
    """
    sentences = []
    labels = []
    current_sentence = []
    current_labels = []

    if not os.path.exists(file_path):
        print(f"Warning: File not found {file_path}")
        return [], []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line_i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) < 2:
                continue
            word = parts[0]
            label = parts[1]
            assert label in LABEL2ID, f"Invalid label {label} in word {word} in file {file_path} (line {line_i+1})"
            current_sentence.append(word)
            current_labels.append(LABEL2ID[label])
            if word.strip() in [".", "!", "?", ""]:
                sentences.append(current_sentence)
                labels.append(current_labels)
                current_sentence = []
                current_labels = []

    if current_sentence:
        sentences.append(current_sentence)
        labels.append(current_labels)

    return sentences, labels


# ==========================================
# 2. Tokenization & Alignment
# ==========================================

def tokenize_and_align_labels(example, tokenizer):
    tokenized = tokenizer(example["tokens"], is_split_into_words=True, truncation=True)
    word_ids = tokenized.word_ids()
    previous_word_idx = None
    labels = []
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(example["labels"][word_idx])
        else:
            labels.append(-100)
        previous_word_idx = word_idx
    tokenized["labels"] = labels
    return tokenized


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [LABEL_LIST[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABEL_LIST[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


def get_model_by_name(model_name: str, label2id, id2label,
                      hidden_dropout_prob: float = 0.5,
                      attention_probs_dropout_prob: float = 0.5):
    if model_name == "FacebookAI/xlm-roberta-base":
        return XLMRobertaForTokenClassification.from_pretrained(
            model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id,
            hidden_dropout_prob=hidden_dropout_prob,
            attention_probs_dropout_prob=attention_probs_dropout_prob)
    elif model_name in ["FacebookAI/roberta-base", "benjamin/roberta-base-wechsel-german"]:
        return AutoModelForTokenClassification.from_pretrained(
            model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id,
            hidden_dropout_prob=hidden_dropout_prob,
            attention_probs_dropout_prob=attention_probs_dropout_prob)
    else:
        return BertForTokenClassification.from_pretrained(
            model_name, num_labels=len(label2id), id2label=id2label, label2id=label2id,
            hidden_dropout_prob=hidden_dropout_prob,
            attention_probs_dropout_prob=attention_probs_dropout_prob)


def get_tokenizer_by_model_name(model_name: str):
    if model_name in ["FacebookAI/roberta-base", "benjamin/roberta-base-wechsel-german"]:
        return RobertaTokenizerFast.from_pretrained('roberta-base', add_prefix_space=True)
    else:
        return AutoTokenizer.from_pretrained(model_name)


# ==========================================
# 3. Inference Logic (Aggregation)
# ==========================================

def aggregate_word_tag(tags) -> str:
    """
    Maps sub-token tags to a single word tag.
    1. All tokens same tag -> that tag.
    2. First token B-IDIOM, all others I-IDIOM -> B-IDIOM.
    3. Otherwise -> INVALID.
    """
    if not tags:
        return "O"
    unique_tags = set(tags)
    if len(unique_tags) == 1:
        return tags[0]
    if tags[0] == "B-IDIOM" and all(t == "I-IDIOM" for t in tags[1:]):
        return "B-IDIOM"
    return "INVALID"


def detailed_inference(model, tokenizer, sentence_words: List[str]) -> List[Dict]:
    tokenized_input = tokenizer(
        sentence_words,
        return_tensors="pt",
        is_split_into_words=True,
        truncation=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model_inputs = {k: v.to(device) for k, v in tokenized_input.items()}
    with torch.no_grad():
        output = model(**model_inputs)
    predictions = torch.argmax(output.logits, dim=2)[0].cpu().numpy()
    word_ids = tokenized_input.word_ids()
    tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"][0])

    structured_result = [{"word": w, "tokens": [], "token_tags": [], "final_tag": None}
                         for w in sentence_words]
    for idx, word_id in enumerate(word_ids):
        if word_id is not None:
            tag = ID2LABEL[predictions[idx]]
            structured_result[word_id]["tokens"].append(tokens[idx])
            structured_result[word_id]["token_tags"].append(tag)
    for item in structured_result:
        item["final_tag"] = aggregate_word_tag(item["token_tags"])
    return structured_result


def get_gold_label_sentences_and_tags(files_paths: List[Path]) -> Dict[str, Dict[str, str]]:
    res = {}
    print(f"{datetime.now()} Loading gold labels from {files_paths}")
    for fpath in files_paths:
        with open(fpath, 'r') as f:
            hard_idioms_list = json.load(f)
            for hard_idiom in hard_idioms_list:
                cur_variant, cur_tokens, cur_tags = [
                    hard_idiom[k] for k in ["variant_sentence", "tokens", "tags"]
                ]
                if all(isinstance(t, list) for t in [cur_tokens, cur_tags]) and cur_variant:
                    res[cur_variant] = dict(zip(cur_tokens, cur_tags, strict=True))
    return res


def already_ran_this_configuration(cur_dir: Path, gold_label_sentences_and_tags) -> bool:
    full_preds_path = cur_dir / PREDICTIONS_OUT_FILENAME
    full_res_path = cur_dir / RESULTS_OUT_FILENAME
    try:
        if not (full_preds_path.exists() and full_res_path.exists()):
            return False
        with open(full_res_path, 'r') as f:
            results = json.load(f)
            if not (["macro_f1", "precision", "recall"] <= list(results.keys())):
                return False
        with open(full_preds_path, 'r') as f:
            preds_dict = json.load(f)
            if not (list(gold_label_sentences_and_tags.keys()) <= list(preds_dict.keys())):
                return False
    except Exception:
        return False
    return True


def score_model_on_sentences(model, tokenizer, input_data: Dict[str, Dict[str, str]],
                              results_out_dir: Path = None):
    """
    Runs inference on {sentence: {word: tag}} input, computes macro-F1/precision/recall,
    and writes aggregated_results.json + sentences_predictions.json to results_out_dir.
    """
    print(f"\nScoring model on {len(input_data)} sentences...")
    y_true, y_pred = [], []
    sentences_predictions = defaultdict(dict)

    for sentence_str, word_tag_map in input_data.items():
        words_in_order = list(word_tag_map.keys())
        cur_preds = detailed_inference(model, tokenizer, words_in_order)
        sentences_predictions[sentence_str]["preds"] = cur_preds
        sentences_predictions[sentence_str]["true_lables"] = word_tag_map

        for res_item in cur_preds:
            y_true.append(word_tag_map[res_item['word']])
            y_pred.append(res_item['final_tag'])

    aggregated_results = {
        "macro_f1": f1_score(y_true, y_pred, average='macro', zero_division=0),
        "precision": precision_score(y_true, y_pred, average='macro', zero_division=0),
        "recall": recall_score(y_true, y_pred, average='macro', zero_division=0)
    }

    if results_out_dir is not None:
        write_results_to_dir(results_out_dir, RESULTS_OUT_FILENAME, aggregated_results)
        write_results_to_dir(results_out_dir, PREDICTIONS_OUT_FILENAME, sentences_predictions)


def get_orig_sent_to_idioms_for_scoring(df: pd.DataFrame):
    return {
        sentence: dict(zip(tokens, tags))
        for sentence, tokens, tags in zip(df["sentence"], df["tokens"], df["tags"])
    }


def get_ds_for_lang(lang: str) -> Dict[str, pd.DataFrame]:
    """
    Returns {"train": train_df, "test": test_df}.

    Train: id10m trainset TSV files.
      Download from https://github.com/Babelscape/ID10M and place under:
      data_generation/id10m_trainset/{lang}/*.tsv

    Test: id10m_fixed cleaned benchmark (committed in this repo).
    """
    lang_trainset_dir = TRAINSET_DIR / lang
    if not lang_trainset_dir.exists():
        raise FileNotFoundError(
            f"Trainset not found at {lang_trainset_dir}.\n"
            f"Download from https://github.com/Babelscape/ID10M "
            f"and place the {lang} TSV files there."
        )
    train_sents, train_label_ids = [], []
    for tsv_file in sorted(lang_trainset_dir.glob("*.tsv")):
        sents, label_ids = parse_tsv_data(tsv_file)
        train_sents.extend(sents)
        train_label_ids.extend(label_ids)
    train_df = pd.DataFrame({"tokens": train_sents})
    train_df["tags"] = train_label_ids
    train_df["tags"] = train_df["tags"].apply(lambda ids: [ID2LABEL[i] for i in ids])

    test_df = pd.read_json(REPO_ROOT / "data_generation" / "id10m_fixed" / f"{lang}.json")

    return {"train": train_df, "test": test_df}


def encode_labels(example):
    example["labels"] = [LABEL2ID[tag] for tag in example["tags"]]
    return example


# ==========================================
# 4. Main Execution Loop
# ==========================================

print(f"{datetime.now()} Start")

for lang_i, lang in enumerate(LANGS):
    cur_data = get_ds_for_lang(lang)
    train_data, test_data = cur_data["train"], cur_data["test"]

    orig_sent_to_idioms = get_orig_sent_to_idioms_for_scoring(test_data)
    gold_label_sentences_and_tags = get_gold_label_sentences_and_tags([DATA_FILE_FOR_LANG[lang]])
    gold_label_sentences_and_tags = gold_label_sentences_and_tags | orig_sent_to_idioms

    for model_i, model_name in enumerate(LANGS_TO_MODELS[lang]):
        print(f"{datetime.now()} Fine-tuning: {lang} ({lang_i+1}/{len(LANGS)}) "
              f"{model_name} ({model_i+1}/{len(LANGS_TO_MODELS[lang])})")

        tokenizer = get_tokenizer_by_model_name(model_name)

        for cur_seed_i, cur_seed in enumerate(SEEDS):
            print(f"{datetime.now()} Seed {cur_seed} ({cur_seed_i+1}/{len(SEEDS)})")

            train_data = train_data.sample(frac=1, random_state=cur_seed).reset_index(drop=True)
            train_dataset = Dataset.from_pandas(train_data[["tokens", "tags"]])
            test_dataset = Dataset.from_pandas(test_data[["tokens", "tags"]])
            dataset_dict = DatasetDict({"train": train_dataset, "test": test_dataset})
            dataset_dict = dataset_dict.map(encode_labels)
            tokenized_datasets = dataset_dict.map(
                lambda x: tokenize_and_align_labels(x, tokenizer), batched=False
            )

            model = get_model_by_name(model_name, label2id=LABEL2ID, id2label=ID2LABEL)
            cur_run_dir = get_cur_run_dir(lang, model_name, cur_seed)

            if already_ran_this_configuration(cur_run_dir, gold_label_sentences_and_tags):
                print(f"{datetime.now()} Already ran — skipping.")
                continue

            data_collator = DataCollatorForTokenClassification(tokenizer)
            training_args = TrainingArguments(
                output_dir=Path(cur_run_dir, "results"),
                eval_strategy="epoch",
                learning_rate=2e-5,
                per_device_train_batch_size=32,
                per_device_eval_batch_size=32,
                num_train_epochs=20,
                weight_decay=0.01,
                warmup_steps=0,
                logging_strategy="epoch",
                logging_dir=Path(cur_run_dir, 'logs'),
                logging_steps=1000,
                save_total_limit=1,
                seed=cur_seed,
                save_strategy="epoch",
                load_best_model_at_end=True,
                dataloader_num_workers=4,
                disable_tqdm=False,
                report_to=[],
                gradient_accumulation_steps=4,
                gradient_checkpointing=False,
                fp16=True,
                optim="adamw_torch",
                adam_beta1=0.9,
                adam_beta2=0.999,
                adam_epsilon=1e-8,
                max_grad_norm=1.0,
                metric_for_best_model="eval_f1",
                greater_is_better=True,
            )

            try:
                for k, v in MODEL_TO_TRAIN_ARGS.get(model_name, {}).items():
                    setattr(training_args, k, v)

                trainer = Trainer(
                    model=model,
                    args=training_args,
                    train_dataset=tokenized_datasets["train"],
                    eval_dataset=tokenized_datasets["test"],
                    tokenizer=tokenizer,
                    data_collator=data_collator,
                    compute_metrics=compute_metrics,
                )

                trainer.train()

                print(f"\n{datetime.now()} Final scoring: {lang} {model_name} seed {cur_seed}")
                score_model_on_sentences(model, tokenizer, gold_label_sentences_and_tags, cur_run_dir)

            except Exception as e:
                print(f"\n{datetime.now()} Exception: {e}")

            finally:
                pass

print(f"\n{datetime.now()} All experiments completed.")

In [ ]:
def get_preds_idioms(preds_list) -> List[str]:
    inside_idiom = False
    pred_idioms, cur_idiom_words = [], []
    for preds_record in preds_list:
        if (final_tag := preds_record["final_tag"]) == "B-IDIOM":
            inside_idiom = True
            cur_idiom_words.append(preds_record["word"])
        elif final_tag == "I-IDIOM" and inside_idiom:
            cur_idiom_words.append(preds_record["word"])
        elif final_tag == "O" and len(cur_idiom_words) > 0:
            pred_idioms.append(" ".join(cur_idiom_words))
            cur_idiom_words = []
            inside_idiom = False
    return pred_idioms


def get_true_idioms_per_lang(lang: str) -> Dict[str, List[str]]:
    true_idioms_per_lang = defaultdict(list)
    with open(DATA_FILE_FOR_LANG[lang], 'r') as f:
        hard_idioms_list = json.load(f)
        for hard_idiom in hard_idioms_list:
            cur_variant, cur_true_idioms = [
                hard_idiom[k] for k in ["variant_sentence", "true_idioms"]
            ]
            true_idioms_per_lang[cur_variant] = cur_true_idioms
    return true_idioms_per_lang


print(f"{datetime.now()} Adding explicit idioms to results — Start")
for cur_lang in LANGS:
    true_idioms_per_lang = get_true_idioms_per_lang(cur_lang)
    for cur_model in LANGS_TO_MODELS[cur_lang]:
        for cur_seed in SEEDS:
            cur_run_dir = get_cur_run_dir(cur_lang, cur_model, cur_seed)
            preds_file = Path(cur_run_dir) / PREDICTIONS_OUT_FILENAME
            if preds_file.exists():
                new_file_data = {}
                with open(preds_file, 'r') as file:
                    preds_file_data = json.load(file)
                    for sent, labels_dict in preds_file_data.items():
                        preds_list = labels_dict.get("preds", [])
                        preds_idioms = get_preds_idioms(preds_list)
                        new_file_data[sent] = copy.deepcopy(labels_dict)
                        new_file_data[sent]["predicted_idioms"] = preds_idioms
                        new_file_data[sent]["true_idioms"] = true_idioms_per_lang.get(sent, [])
                write_results_to_dir(
                    Path(cur_run_dir), PREDICTIONS_OUT_FILENAME_PLUS_EXPLICIT_IDIOMS, new_file_data
                )

print(f"\n{datetime.now()} Finished adding explicit idioms")

In [ ]:
print(f"{datetime.now()} FIN")